In [5]:
tabla="cbnat10"
dim="dim_numaten"

In [6]:

#################################################################################################################################################################################################################################################################################
#CONEXION AL ORACLE

import oracledb
import pandas as pd
from sqlalchemy import create_engine
import sys
from sqlalchemy import create_engine
from sqlalchemy import text
import sys
from decouple import config

oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb
un = config("USER4_BDI_POSTGRES")
pw = config("PASS4_BDI_POSTGRES")
hostname=config("HOST4_BDI_POSTGRES")
service_name="WNET"
port = 1527

engine2 = create_engine(f'oracle://{un}:{pw}@',connect_args={
        "host": hostname,
        "port": port,
        "service_name": service_name
    }
)

connection2 = engine2.connect()

base2 = pd.read_sql_query(f"SELECT * FROM {tabla}", con=connection2)

connection2.close()




In [7]:
base2

,numatecod,numatedes,numatedescor
0,0,NO NUMERO,None
1,1,PRIMERA,1RA
2,2,SEGUNDA,2DA
3,3,TERCERA,3RA
4,4,CUARTA,4TA
5,5,QUINTA,5TA
6,U,UNICA,None
7,R1,PRIMERA REFUERZO,None
8,R2,SEGUNDA REFUERZO,None
9,0M,NO NUMERO,None


In [8]:
base2.columns

Index(['numatecod', 'numatedes', 'numatedescor'], dtype='object')

In [9]:

#################################################################################################################################################################################################################################################################################
#CREAMOS LA TABLA TEMPORAL Y LA SUBIMOS AL POSTGRES SQL

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()



#connection3.execute('CREATE TEMPORARY TABLE tmp_cmdia10 ()')
base2.to_sql(name=f'{tabla}', con=engine3, if_exists='replace', index=False)


21

In [10]:
DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dw_essalud"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena4  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

engine4 = create_engine(cadena4)
connection4 = engine4.connect()


In [11]:
base2.to_sql(name=f'tmp_{tabla}', con=engine4, if_exists='replace', index=False)


21

In [17]:
query=f"""
INSERT INTO {dim} (cod_numaten, des_numaten) 
SELECT numatecod,numatedes
FROM tmp_{tabla};

DROP TABLE tmp_{tabla}
"""
c1= text(query)
connection4.execute(c1)
connection4.close()